# E07 / E10 — Valutazione su ExDark (NIQE + BRISQUE)

Valuta entrambi i modelli sul dataset ExDark con metriche no-reference.

| | |
|---|---|
| Dataset | ExDark (7363 immagini, 12 classi) |
| Metriche | NIQE (lower=better), BRISQUE (lower=better) |
| Output | `baseline_exdark.csv`, `variant_exdark.csv` |
| Hardware target | Kaggle T4 (16 GB VRAM) |

In [ ]:
# ── Installazione dipendenze ──────────────────────────────────────────────
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", *pkgs, "-q"])

for pkg, imp in [("piq>=0.8.0", "piq"), ("pyiqa>=0.1.0", "pyiqa"),
                 ("tqdm>=4.65.0", "tqdm"), ("torchinfo>=1.8.0", "torchinfo")]:
    try:
        __import__(imp)
    except ImportError:
        _pip(pkg)

print("Dipendenze OK")

In [ ]:
# ── Import e rilevamento ambiente ─────────────────────────────────────────
import os, sys
from pathlib import Path
import torch

ON_KAGGLE = Path("/kaggle/working").exists()

if ON_KAGGLE:
    PROJECT_ROOT = Path("/kaggle/input/datasets/mattiacagnetta/low-light-src")
    OUTPUT_ROOT  = Path("/kaggle/working")
else:
    PROJECT_ROOT = Path("..").resolve()
    OUTPUT_ROOT  = PROJECT_ROOT

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"ON_KAGGLE    : {ON_KAGGLE}")
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Costanti — aggiusta i path se necessario ──────────────────────────────
if ON_KAGGLE:
    # Dataset ExDark — modifica il nome slug se diverso
    EXDARK_ROOT  = Path("/kaggle/input/datasets/mattiacagnetta/exdark/ExDark")
    # Checkpoints
    CKPT_ROOT    = Path("/kaggle/input/datasets/mattiacagnetta/low-light-checkpoints")
else:
    EXDARK_ROOT  = PROJECT_ROOT / "data" / "ExDark"
    CKPT_ROOT    = PROJECT_ROOT / "checkpoint"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = OUTPUT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_CKPT  = CKPT_ROOT / "baseline"  / "best.pt"
ATTENTION_CKPT = CKPT_ROOT / "attention" / "best.pt"

# Diagnostica
for label, p in [
    ("ExDark root",   EXDARK_ROOT),
    ("baseline best.pt",  BASELINE_CKPT),
    ("attention best.pt", ATTENTION_CKPT),
]:
    status = "\u2713" if p.exists() else "\u2717 NON TROVATO"
    print(f"  {status}  {label}: {p}")

# Conta immagini ExDark
if EXDARK_ROOT.exists():
    exts = {".png", ".jpg", ".jpeg", ".bmp"}
    imgs = [p for p in EXDARK_ROOT.rglob("*") if p.suffix.lower() in exts]
    print(f"\nImmagini ExDark: {len(imgs)}")

In [ ]:
# ── E07 — Baseline su ExDark ──────────────────────────────────────────────
from src.models  import UNetBaseline
from src.evaluate import Evaluator, load_model_from_checkpoint

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

baseline = load_model_from_checkpoint(
    BASELINE_CKPT, UNetBaseline(base_channels=32), device=device
)
ev_base = Evaluator(baseline, device=device, batch_size=16)

df_base = ev_base.evaluate_unpaired(
    image_dir  = EXDARK_ROOT,
    glob       = "**/*",
    output_csv = RESULTS_DIR / "baseline_exdark.csv",
)

data = df_base[~df_base["stem"].astype(str).str.startswith("**")]
mean = df_base[df_base["stem"] == "**mean**"].iloc[0]
std  = df_base[df_base["stem"] == "**std**"].iloc[0]
print(f"\nBaseline su ExDark ({len(data)} immagini):")
print(f"  NIQE    mean={mean.niqe:.2f}  std={std.niqe:.2f}  (lower=better)")
print(f"  BRISQUE mean={mean.brisque:.2f}  std={std.brisque:.2f}  (lower=better)")

In [ ]:
# ── E10 — Attention su ExDark ─────────────────────────────────────────────
from src.models import UNetAttention

attention = load_model_from_checkpoint(
    ATTENTION_CKPT, UNetAttention(base_channels=32), device=device
)
ev_attn = Evaluator(attention, device=device, batch_size=16)

df_attn = ev_attn.evaluate_unpaired(
    image_dir  = EXDARK_ROOT,
    glob       = "**/*",
    output_csv = RESULTS_DIR / "variant_exdark.csv",
)

data = df_attn[~df_attn["stem"].astype(str).str.startswith("**")]
mean = df_attn[df_attn["stem"] == "**mean**"].iloc[0]
std  = df_attn[df_attn["stem"] == "**std**"].iloc[0]
print(f"\nAttention su ExDark ({len(data)} immagini):")
print(f"  NIQE    mean={mean.niqe:.2f}  std={std.niqe:.2f}  (lower=better)")
print(f"  BRISQUE mean={mean.brisque:.2f}  std={std.brisque:.2f}  (lower=better)")

In [ ]:
# ── Riepilogo e download ───────────────────────────────────────────────────
import pandas as pd

def get_mean(df, col):
    return df[df["stem"] == "**mean**"].iloc[0][col]

print("=" * 55)
print("RIEPILOGO ExDark — NIQE + BRISQUE (lower=better)")
print("=" * 55)
print(f"  {'':20} {'NIQE':>10} {'BRISQUE':>10}")
print(f"  {'Baseline':20} {get_mean(df_base,'niqe'):>10.2f} {get_mean(df_base,'brisque'):>10.2f}")
print(f"  {'Attention':20} {get_mean(df_attn,'niqe'):>10.2f} {get_mean(df_attn,'brisque'):>10.2f}")

# Zip per download su Kaggle
if ON_KAGGLE:
    import shutil
    from IPython.display import FileLink, display
    zip_path = "/kaggle/working/exdark_results.zip"
    shutil.make_archive(
        "/kaggle/working/exdark_results", "zip",
        root_dir=str(RESULTS_DIR.parent),
        base_dir="results",
    )
    print(f"\nZip pronto:")
    display(FileLink("exdark_results.zip"))